# Physics-informed graph neural network for multi-strain forecasting

This notebook combines a neural feature encoder, a learned history-dependent prediction graph, graph convolution, GRU temporal encoding and multi-step decoding, time-varying transmission rates $\beta_i^k(t)$, and a shared positive dispersal coefficient $d_0$. Training uses four terms: forecast error, the complete $SI^3R$ equation residual, temporal parameter regularization, and an ensemble Kalman teacher.

## Input data

- The workbook contains NSW, Vic, Qld, SA, WA, Tas, and NT. ACT is excluded.
- Required columns are year_week, infection_count, strain_type, tp_m_day, t2m_K, and d2m_K.
- The strain order is H3N2, H1N1, and B, mapped to A(H3N2), A(H1N1), and B.
- The supplied workbook spans 2013-W01 to 2017-W52 (261 weekly observations).
- Missing region-week-strain rows are filled with zero cases. Weather gaps are interpolated over time. Normalization statistics use only the training period.

## Configuration provenance

The numerical configuration is: s03_h12_log35, a 12-week history, 32-dimensional hidden representations, dropout 0.18, mixed linear/logarithmic data loss, observation-anchored residual predictions, a physics constraint, and an intermittently evaluated Kalman teacher.

In [ ]:
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
import copy
import os
import json
import math
from time import perf_counter
import random
import secrets
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)

def resolve_project_dir(required_filename):
    """Find a co-located input workbook without machine-specific paths."""
    override = os.environ.get("SIKR_PROJECT_DIR")
    if override:
        project_dir = Path(override).expanduser().resolve()
        if not (project_dir / required_filename).is_file():
            raise FileNotFoundError(
                f"SIKR_PROJECT_DIR does not contain {required_filename}: {project_dir}"
            )
        return project_dir

    candidates = []
    if globals().get("__file__"):
        candidates.append(Path(__file__).resolve().parent)
    working_dir = Path.cwd().resolve()
    candidates.extend([working_dir, *working_dir.parents])
    for candidate in dict.fromkeys(candidates):
        if (candidate / required_filename).is_file():
            return candidate
    raise FileNotFoundError(
        f"Cannot locate {required_filename}. Keep the workbook beside this notebook "
        "and start Jupyter from that folder, or set SIKR_PROJECT_DIR to the folder."
    )




## 1. Configuration and fixed dynamical parameters

The state order is $[S,I^{H3N2},I^{H1N1},I^B,R]$. Among the epidemiological parameters, only $\beta_i^k(t)$ and $d_0$ are learned. The remaining epidemiological parameters are fixed buffers; the neural-network weights are also optimized.

In [ ]:
@dataclass
class Config:
    data_path: str = "FinalDataAustralia2013-2017.xlsx"
    history: int = 12
    horizon: int = 4
    split_ratio: tuple = (3, 1, 1)
    batch_size: int = 16
    num_workers: int = 0

    d_model: int = 32
    node_emb_dim: int = 4
    dropout: float = 0.18
    graph_mix_init: float = 0.25
    log_data_weight: float = 0.35
    residual_weight: float = 0.85
    max_log_change: float = 1.40

    beta_scale: float = 1.0
    d0_scale: float = 1.0e-2
    dt: float = 1.0

    lr_neural: float = 1.0e-3
    lr_physics: float = 1.0e-4
    weight_decay: float = 1.0e-5
    grad_clip: float = 1.0
    epochs: int = 90
    patience: int = 18

    lambda_data: float = 1.0
    lambda_eq_max: float = 0.05
    lambda_para: float = 3.0e-3
    lambda_kalman: float = 1.0e-2
    physics_warmup_start: int = 10
    physics_warmup_epochs: int = 30

    ensemble_size: int = 8
    process_noise_fraction: float = 0.01
    observation_noise_fraction: float = 0.10
    kalman_every_n_batches: int = 4
    random_seed_runs: int = 10  # Generate and record a new seed for each run.

#     run_training: bool = False   # Set True to execute full training.
    run_training: bool = True   # Set True to execute full training.
    smoke_test: bool = False

CFG = Config()
PROJECT_DIR = resolve_project_dir(CFG.data_path)
print("Project directory:", PROJECT_DIR)

# Node order matches the fixed parameters; sheet names retain workbook capitalization.
STATES = ["NSW", "Vic", "QLD", "SA", "WA", "TAS", "NT"]
SHEET_MAP = {"NSW": "NSW", "Vic": "Vic", "QLD": "Qld", "SA": "SA", "WA": "WA", "TAS": "Tas", "NT": "NT"}
STRAINS = ["H3N2", "H1N1", "B"]
STRAIN_MAP = {"H3N2": "A(H3N2)", "H1N1": "A(H1N1)", "B": "B"}
WEATHER_COLUMNS = ["tp_m_day", "t2m_K", "d2m_K"]

B_FIXED = np.array([1856.40, 1492.85, 1193.58, 379.47, 671.70, 111.50, 75.95], dtype=np.float32)
OMEGA_FIXED = np.full(7, 6.9237e-3, dtype=np.float32)
MU_FIXED = np.array([1.31374e-4, 1.22434e-4, 1.18322e-4, 1.51732e-4,
                     1.07574e-4, 1.69783e-4, 8.72950e-5], dtype=np.float32)
ALPHA_FIXED = np.array([6.96e-4, 2.70e-4, 4.64e-4], dtype=np.float32)
Q_FIXED = np.array([0.252, 0.269, 0.261], dtype=np.float32)
GAMMA_FIXED = np.array([1.728, 1.720, 1.720], dtype=np.float32)

assert len(STATES) == len(B_FIXED) == len(MU_FIXED) == len(OMEGA_FIXED) == 7
assert len(STRAINS) == len(ALPHA_FIXED) == len(Q_FIXED) == len(GAMMA_FIXED) == 3

PARAMETER_TABLE = pd.DataFrame({
    "state": STATES, "B_person_per_week": B_FIXED,
    "mu_per_week": MU_FIXED, "omega_per_week": OMEGA_FIXED,
})
TYPE_PARAMETER_TABLE = pd.DataFrame({
    "strain": STRAINS, "alpha_per_week": ALPHA_FIXED,
    "q": Q_FIXED, "gamma_per_week": GAMMA_FIXED,
})
display(PARAMETER_TABLE)
display(TYPE_PARAMETER_TABLE)


## 2. Load and reshape the workbook

The long-format input is converted to regular case and weather tensors, each of shape $T\times7\times3$. A continuous ISO-week index is constructed before missing case combinations are filled with zero. Weather is aggregated within each region-week and interpolated over time.


In [ ]:
def iso_week_to_monday(year_week: str) -> pd.Timestamp:
    s = str(year_week).replace(".0", "").zfill(6)
    return pd.Timestamp(date.fromisocalendar(int(s[:4]), int(s[4:]), 1))


def load_excel_tensor(cfg: Config):
    path = PROJECT_DIR / cfg.data_path
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    raw = {}
    all_dates = set()
    audit_rows = []
    for state in STATES:
        df = pd.read_excel(path, sheet_name=SHEET_MAP[state])
        expected = {"year_week", "infection_count", "strain_type", *WEATHER_COLUMNS}
        missing_columns = expected.difference(df.columns)
        if missing_columns:
            raise ValueError(f"{state} missing columns: {sorted(missing_columns)}")
        df = df[list(expected)].copy()
        df["date"] = df["year_week"].map(iso_week_to_monday)
        if df.duplicated(["date", "strain_type"]).any():
            raise ValueError(f"{state} contains duplicate week-strain keys")
        raw[state] = df
        all_dates.update(df["date"].tolist())
        audit_rows.append({"state": state, "raw_rows": len(df), "observed_weeks": df["date"].nunique()})

    dates = pd.date_range(min(all_dates), max(all_dates), freq="W-MON")
    T, M, K = len(dates), len(STATES), len(STRAINS)
    cases = np.zeros((T, M, K), dtype=np.float32)
    weather = np.zeros((T, M, len(WEATHER_COLUMNS)), dtype=np.float32)

    for i, state in enumerate(STATES):
        df = raw[state]
        case_pivot = (df.pivot(index="date", columns="strain_type", values="infection_count")
                        .reindex(index=dates, columns=[STRAIN_MAP[s] for s in STRAINS])
                        .fillna(0.0))
        weather_week = (df.groupby("date", as_index=True)[WEATHER_COLUMNS]
                          .mean()
                          .reindex(dates)
                          .interpolate(method="time", limit_direction="both"))
        if weather_week.isna().any().any():
            raise ValueError(f"{state} still has missing weather values after interpolation")
        cases[:, i, :] = case_pivot.to_numpy(dtype=np.float32)
        weather[:, i, :] = weather_week.to_numpy(dtype=np.float32)

    audit = pd.DataFrame(audit_rows)
    audit["complete_weeks"] = T
    audit["completed_week_strain_rows"] = T * K
    return dates, cases, weather, audit


DATES, CASES_RAW, WEATHER_RAW, DATA_AUDIT = load_excel_tensor(CFG)
print("Date range:", DATES[0].date(), "to", DATES[-1].date(), "| weeks:", len(DATES))
print("Case tensor:", CASES_RAW.shape, "| weather tensor:", WEATHER_RAW.shape)
display(DATA_AUDIT)



## 3. Chronological split, normalization, and sliding windows

The target periods are split chronologically in a 3:1:1 ratio. Validation and test windows may use observations preceding their forecast origin, but never future targets as input. Case features are log1p-transformed and standardized per region-strain; weather features are standardized per region-variable. All scaling statistics come from the training period.


In [ ]:
def safe_std(x, axis=0, keepdims=False):
    s = np.std(x, axis=axis, keepdims=keepdims)
    return np.where(s < 1e-6, 1.0, s)


def prepare_features(dates, cases, weather, cfg: Config):
    T, M, K = cases.shape
    train_end = int(T * cfg.split_ratio[0] / sum(cfg.split_ratio))
    val_end = int(T * (cfg.split_ratio[0] + cfg.split_ratio[1]) / sum(cfg.split_ratio))

    log_cases = np.log1p(cases)
    case_mean = log_cases[:train_end].mean(axis=0, keepdims=True)
    case_std = safe_std(log_cases[:train_end], axis=0, keepdims=True)
    case_z = (log_cases - case_mean) / case_std

    weather_mean = weather[:train_end].mean(axis=0, keepdims=True)
    weather_std = safe_std(weather[:train_end], axis=0, keepdims=True)
    weather_z = (weather - weather_mean) / weather_std

    # isocalendar() may return a named result or a tuple across Python/pandas versions.
    # Positional index [1] supports both representations.
    iso_week = np.array([int(pd.Timestamp(d).isocalendar()[1]) for d in dates], dtype=np.float32)
    week_sin = np.sin(2 * np.pi * iso_week / 52.1775)[:, None, None]
    week_cos = np.cos(2 * np.pi * iso_week / 52.1775)[:, None, None]
    time_features = np.concatenate([
        np.broadcast_to(week_sin, (T, M, 1)),
        np.broadcast_to(week_cos, (T, M, 1)),
    ], axis=-1).astype(np.float32)

    static = np.stack([B_FIXED, MU_FIXED, OMEGA_FIXED], axis=-1)
    static_mean = static.mean(axis=0, keepdims=True)
    static_std = safe_std(static, axis=0, keepdims=True)
    static_z = ((static - static_mean) / static_std).astype(np.float32)
    static_z = np.broadcast_to(static_z[None, :, :], (T, M, 3))

    features = np.concatenate([case_z, weather_z, time_features, static_z], axis=-1).astype(np.float32)
    case_scale = np.quantile(cases[:train_end], 0.95, axis=0).astype(np.float32)
    case_scale = np.maximum(case_scale, 1.0)

    scaler = {
        "case_mean": case_mean, "case_std": case_std,
        "weather_mean": weather_mean, "weather_std": weather_std,
        "case_scale": case_scale,
    }
    return features, scaler, train_end, val_end


FEATURES, SCALER, TRAIN_END, VAL_END = prepare_features(DATES, CASES_RAW, WEATHER_RAW, CFG)
print("Feature tensor:", FEATURES.shape, "| train_end:", TRAIN_END, "| val_end:", VAL_END)


class WindowDataset(Dataset):
    def __init__(self, features, cases, dates, origins, history, horizon):
        self.features = torch.as_tensor(features, dtype=torch.float32)
        self.cases = torch.as_tensor(cases, dtype=torch.float32)
        self.dates = dates
        self.origins = np.asarray(origins, dtype=np.int64)
        self.history = history
        self.horizon = horizon

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, index):
        t = int(self.origins[index])
        x_hist = self.features[t - self.history + 1:t + 1]
        last_cases = self.cases[t]
        y_future = self.cases[t + 1:t + self.horizon + 1]
        return x_hist, last_cases, y_future, t


def make_origins(T, history, horizon, train_end, val_end):
    candidates = np.arange(history - 1, T - horizon)
    target_start = candidates + 1
    target_end = candidates + horizon
    train = candidates[target_end < train_end]
    val = candidates[(target_start >= train_end) & (target_end < val_end)]
    test = candidates[target_start >= val_end]
    return train, val, test


TRAIN_ORIGINS, VAL_ORIGINS, TEST_ORIGINS = make_origins(
    len(DATES), CFG.history, CFG.horizon, TRAIN_END, VAL_END
)

train_ds = WindowDataset(FEATURES, CASES_RAW, DATES, TRAIN_ORIGINS, CFG.history, CFG.horizon)
val_ds = WindowDataset(FEATURES, CASES_RAW, DATES, VAL_ORIGINS, CFG.history, CFG.horizon)
test_ds = WindowDataset(FEATURES, CASES_RAW, DATES, TEST_ORIGINS, CFG.history, CFG.horizon)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

print("Window counts train/val/test:", len(train_ds), len(val_ds), len(test_ds))
print("Target-period boundaries:", DATES[TRAIN_END].date(), DATES[VAL_END].date())


## 4. Physical graph and Laplacian

The differential equations use a fixed Laplacian $L=W-D$. The prediction GNN learns history-dependent adjacency weights from the prior. Those weights do not replace the fixed physical graph, keeping its scale separate from $d_0$.


In [ ]:
def build_default_physical_graph(n_nodes: int) -> torch.Tensor:
    # Use a symmetric nonnegative complete-graph prior with zero diagonal and unit row sums.
    W = torch.ones(n_nodes, n_nodes, dtype=torch.float32) - torch.eye(n_nodes, dtype=torch.float32)
    W = W / W.sum(dim=1, keepdim=True)
    return 0.5 * (W + W.T)


W_PHYS = build_default_physical_graph(len(STATES))
D_PHYS = torch.diag(W_PHYS.sum(dim=1))
L_PHYS = W_PHYS - D_PHYS

assert torch.allclose(W_PHYS, W_PHYS.T, atol=1e-7)
assert torch.allclose(torch.diag(W_PHYS), torch.zeros(len(STATES)), atol=1e-7)
assert torch.allclose(L_PHYS @ torch.ones(len(STATES)), torch.zeros(len(STATES)), atol=1e-6)
eigvals = torch.linalg.eigvalsh(L_PHYS)
print("Largest eigenvalue of L (expected <= 0):", float(eigvals.max()))
display(pd.DataFrame(W_PHYS.numpy(), index=STATES, columns=STATES))


## 5. Neural encoding, graph convolution, and temporal encoding

The pure neural encoder combines the input features with learned node embeddings. A learned adjacency matrix is mixed with the prior graph and used for graph convolution. A GRU then encodes temporal dependence.

In [ ]:
class NeuralEncoder(nn.Module):
    """Pure neural node encoder"""
    def __init__(self, input_dim, d_model, n_nodes, node_emb_dim, dropout):
        super().__init__()
        self.node_embedding = nn.Embedding(n_nodes, node_emb_dim)
        self.neural = nn.Sequential(
            nn.Linear(input_dim + node_emb_dim, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.GELU(),
        )

    def forward(self, x):
        # x: [B,H,M,F]
        batch_size, history, nodes, _ = x.shape
        node_ids = torch.arange(nodes, device=x.device)
        embedding = self.node_embedding(node_ids)[None, None].expand(batch_size, history, nodes, -1)
        return self.neural(torch.cat([x, embedding], dim=-1))


class AdaptiveGraphConv(nn.Module):
    def __init__(self, d_model, base_W, mix_init=0.25, dropout=0.1):
        super().__init__()
        self.register_buffer("base_W", base_W)
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.message = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        mix_init = min(max(mix_init, 1e-4), 1 - 1e-4)
        self.mix_logit = nn.Parameter(torch.tensor(math.log(mix_init / (1 - mix_init)), dtype=torch.float32))

    def forward(self, h):
        # h: [B,H,M,D]
        q = self.query(h)
        score = torch.einsum("bhid,bhjd->bhij", q, q) / math.sqrt(q.shape[-1])
        mask = 1.0 - torch.eye(q.shape[-2], device=h.device)[None, None]
        A_dyn = torch.sigmoid(score) * mask
        A_dyn = 0.5 * (A_dyn + A_dyn.transpose(-1, -2))
        # Preserve symmetry and scale total graph weight to M (mean row sum = 1).
        A_dyn = A_dyn * q.shape[-2] / A_dyn.sum(dim=(-1, -2), keepdim=True).clamp_min(1e-6)

        mix = torch.sigmoid(self.mix_logit)
        A = (1.0 - mix) * self.base_W[None, None] + mix * A_dyn
        eye = torch.eye(A.shape[-1], device=h.device)[None, None]
        A_tilde = A + eye
        degree = A_tilde.sum(dim=-1).clamp_min(1e-6)
        A_norm = A_tilde / torch.sqrt(degree.unsqueeze(-1) * degree.unsqueeze(-2))
        aggregated = torch.einsum("bhij,bhjd->bhid", A_norm, h)
        out = self.norm(h + self.dropout(F.gelu(self.message(aggregated))))
        return out, A, mix


class TemporalEncoder(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(d_model, d_model, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h):
        Bn, H, M, D = h.shape
        seq = h.permute(0, 2, 1, 3).reshape(Bn * M, H, D)
        out, _ = self.gru(seq)
        context = self.dropout(out[:, -1]).reshape(Bn, M, D)
        return context


## 6. Multi-step decoding and dynamical parameter heads

A shared parameter head produces $\beta_i^k(t)$ along the decoded trajectory, rather than learning an unrelated free parameter for every observed week. The same head can therefore produce rates at forecast times. The shared coefficient $d_0$ is a positive trainable scalar.


In [ ]:
def inverse_softplus(x: float) -> float:
    return math.log(math.expm1(x))


class HorizonDecoder(nn.Module):
    def __init__(self, d_model, max_horizon, dropout=0.1):
        super().__init__()
        self.horizon_embedding = nn.Embedding(max_horizon + 1, d_model)
        self.cell = nn.GRUCell(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, context, horizon):
        Bn, M, D = context.shape
        q = context
        outputs = [q]
        for h in range(1, horizon + 1):
            emb = self.horizon_embedding.weight[h][None, None].expand(Bn, M, D)
            q = self.cell(emb.reshape(Bn * M, D), q.reshape(Bn * M, D)).reshape(Bn, M, D)
            outputs.append(self.dropout(q))
        return torch.stack(outputs, dim=1)  # [B,P+1,M,D]


class StateHead(nn.Module):
    def __init__(self, d_model, n_ref, case_scale):
        super().__init__()
        self.proj = nn.Linear(d_model, 6)  # N scale, two S/R shares, and three infectious components
        self.register_buffer("n_ref", n_ref)
        self.register_buffer("case_scale", case_scale)

    def forward(self, q_seq, last_cases):
        raw = self.proj(q_seq)
        n_hat = self.n_ref[None, None, :] * (0.8 + 0.4 * torch.sigmoid(raw[..., 0]))
        sr_share = torch.softmax(raw[..., 1:3], dim=-1)

        # Combine direct predictions with log-residual predictions anchored to the latest observations.
        direct = F.softplus(raw[..., 3:6]) * self.case_scale[None, None]
        base_log = torch.log1p(last_cases)[:, None]
        residual_log = base_log + CFG.max_log_change * torch.tanh(raw[..., 3:6])
        residual = torch.expm1(residual_log.clamp(min=0.0, max=20.0))
        i_pred = CFG.residual_weight * residual + (1.0 - CFG.residual_weight) * direct
        i_all = torch.cat([last_cases[:, None], i_pred[:, 1:]], dim=1)

        remaining = (n_hat - i_all.sum(dim=-1)).clamp_min(0.01 * self.n_ref[None, None, :])
        s_hat = remaining * sr_share[..., 0]
        r_hat = remaining * sr_share[..., 1]
        return torch.cat([s_hat[..., None], i_all, r_hat[..., None]], dim=-1)


class BetaHead(nn.Module):
    def __init__(self, d_model, n_nodes, n_types, beta_scale):
        super().__init__()
        self.shared = nn.Linear(d_model, n_types)
        self.node_type_bias = nn.Parameter(torch.zeros(n_nodes, n_types))
        self.beta_scale = beta_scale
        nn.init.normal_(self.shared.weight, mean=0.0, std=0.02)
        nn.init.constant_(self.shared.bias, 0.5)

    def forward(self, q_seq):
        eta = self.shared(q_seq) + self.node_type_bias[None, None]
        return self.beta_scale * (F.softplus(eta) + 1e-6)


class PhysicsGNN(nn.Module):
    def __init__(self, cfg, input_dim, base_W, case_scale):
        super().__init__()
        M, K = len(STATES), len(STRAINS)
        n_ref = torch.as_tensor(B_FIXED / MU_FIXED, dtype=torch.float32)
        self.encoder = NeuralEncoder(
            input_dim=input_dim, d_model=cfg.d_model, n_nodes=M,
            node_emb_dim=cfg.node_emb_dim, dropout=cfg.dropout,
        )
        self.graph = AdaptiveGraphConv(cfg.d_model, base_W, cfg.graph_mix_init, cfg.dropout)
        self.temporal = TemporalEncoder(cfg.d_model, cfg.dropout)
        self.decoder = HorizonDecoder(cfg.d_model, cfg.horizon, cfg.dropout)
        self.state_head = StateHead(cfg.d_model, n_ref, case_scale)
        self.beta_head = BetaHead(cfg.d_model, M, K, cfg.beta_scale)
        self.raw_d0 = nn.Parameter(torch.tensor(inverse_softplus(0.10), dtype=torch.float32))
        self.d0_scale = cfg.d0_scale
        self.horizon = cfg.horizon

    def d0(self):
        return self.d0_scale * (F.softplus(self.raw_d0) + 1e-8)

    def forward(self, x_hist, last_cases):
        encoded = self.encoder(x_hist)
        spatial, adjacency, graph_mix = self.graph(encoded)
        context = self.temporal(spatial)
        q_seq = self.decoder(context, self.horizon)
        states = self.state_head(q_seq, last_cases)
        beta = self.beta_head(q_seq)
        return {
            "states": states, "beta": beta, "d0": self.d0(),
            "adjacency": adjacency, "graph_mix": graph_mix,
        }


## 7. SI$^3$R dynamics and equation residual

The diffusion terms use the negative-semidefinite Laplacian $L=W-D$ with a positive coefficient. The weekly equation residual is evaluated using the trapezoidal rule:

$$r_h=\frac{X_{h+1}-X_h}{\Delta t}-\frac{F(X_h,\beta_h,d_0)+F(X_{h+1},\beta_{h+1},d_0)}{2}.$$


In [ ]:
class SI3RPhysics(nn.Module):
    def __init__(self, L_phys, case_scale):
        super().__init__()
        self.register_buffer("L", L_phys)
        self.register_buffer("B", torch.as_tensor(B_FIXED))
        self.register_buffer("mu", torch.as_tensor(MU_FIXED))
        self.register_buffer("omega", torch.as_tensor(OMEGA_FIXED))
        self.register_buffer("alpha", torch.as_tensor(ALPHA_FIXED))
        self.register_buffer("gamma", torch.as_tensor(GAMMA_FIXED))
        self.register_buffer("q", torch.as_tensor(Q_FIXED))
        case_scale = torch.as_tensor(case_scale, dtype=torch.float32)
        # Scale the unobserved S/R states by the reference population B/mu.
        # Scale infectious-state residuals by the training-set 95th percentile so large S/R values do not dominate.
        n_ref = self.B / self.mu
        self.register_buffer("residual_scale", torch.cat([
            n_ref[:, None], case_scale, n_ref[:, None]
        ], dim=-1).clamp_min(1.0))

    def rhs(self, state, beta, d0):
        # state [..., M, 5], beta [..., M, 3]
        S = state[..., 0]
        I = state[..., 1:4]
        R = state[..., 4]
        N = state.sum(dim=-1).clamp_min(1.0)

        lap_S = torch.einsum("ij,...j->...i", self.L, S)
        lap_I = torch.einsum("ij,...jk->...ik", self.L, I)
        lap_R = torch.einsum("ij,...j->...i", self.L, R)

        incidence = (S / N)[..., None] * beta * I
        dS = (self.B - self.mu * S - incidence.sum(dim=-1)
              + self.omega * R + d0 * lap_S)
        removal = self.mu[..., None] + self.alpha[None, :] + self.gamma[None, :]
        dI = incidence - removal * I + d0 * self.q[None, :] * lap_I
        dR = ((self.gamma[None, :] * I).sum(dim=-1)
              - (self.mu + self.omega) * R + d0 * lap_R)
        return torch.cat([dS[..., None], dI, dR[..., None]], dim=-1)

    def rk4_step(self, state, beta, d0, dt=1.0):
        k1 = self.rhs(state, beta, d0)
        k2 = self.rhs((state + 0.5 * dt * k1).clamp_min(0.0), beta, d0)
        k3 = self.rhs((state + 0.5 * dt * k2).clamp_min(0.0), beta, d0)
        k4 = self.rhs((state + dt * k3).clamp_min(0.0), beta, d0)
        return (state + dt * (k1 + 2*k2 + 2*k3 + k4) / 6.0).clamp_min(0.0)

    def equation_loss(self, states, beta, d0, dt=1.0):
        f = self.rhs(states, beta, d0)
        residual = ((states[:, 1:] - states[:, :-1]) / dt
                    - 0.5 * (f[:, 1:] + f[:, :-1]))
        scaled = residual / self.residual_scale[None, None]
        return scaled.square().mean(), residual


PHYSICS = SI3RPhysics(L_PHYS, SCALER["case_scale"]).to(DEVICE)


## 8. Ensemble Kalman teacher

The nonlinear system has 35 state components. An ensemble Kalman analysis avoids explicitly forming a 35-by-35 Jacobian. The teacher assimilates only the observations at the forecast origin and then propagates without future observations.

The teacher returns a mean and variance used in the variance-weighted loss $L_K$. Its computations are detached from autograd and introduce no additional trainable parameters.


In [ ]:
@torch.no_grad()
def ensemble_kalman_teacher(physics, state0, y0, beta_seq, d0, cfg: Config):
    # state0 [B,M,5], y0 [B,M,3], beta_seq [B,P+1,M,3]
    Bn, M, C = state0.shape
    E, D, O = cfg.ensemble_size, M * C, M * 3
    n_ref = torch.as_tensor(B_FIXED / MU_FIXED, device=state0.device)
    case_scale = physics.residual_scale[:, 1:4]
    state_noise = torch.cat([
        (0.005 * n_ref)[:, None],
        (0.20 * case_scale),
        (0.005 * n_ref)[:, None],
    ], dim=-1)

    ensemble = state0[:, None] + torch.randn(Bn, E, M, C, device=state0.device) * state_noise[None, None]
    ensemble = ensemble.clamp_min(0.0)

    X = ensemble.reshape(Bn, E, D)
    Y = ensemble[..., 1:4].reshape(Bn, E, O)
    Xc = X - X.mean(dim=1, keepdim=True)
    Yc = Y - Y.mean(dim=1, keepdim=True)
    Pxy = torch.einsum("bed,beo->bdo", Xc, Yc) / max(E - 1, 1)
    Pyy = torch.einsum("beo,bep->bop", Yc, Yc) / max(E - 1, 1)

    obs_std = (cfg.observation_noise_fraction * case_scale).clamp_min(1.0).reshape(O)
    Rmat = torch.diag(obs_std.square())[None].expand(Bn, -1, -1)
    gain = torch.linalg.solve(Pyy + Rmat, Pxy.transpose(-1, -2)).transpose(-1, -2)
    perturbed_obs = y0.reshape(Bn, 1, O) + torch.randn(Bn, E, O, device=state0.device) * obs_std[None, None]
    X = X + torch.einsum("bdo,beo->bed", gain, perturbed_obs - Y)
    ensemble = X.reshape(Bn, E, M, C).clamp_min(0.0)

    means, variances = [], []
    process_scale = cfg.process_noise_fraction * state_noise
    for h in range(cfg.horizon):
        beta_h = beta_seq[:, h][:, None].expand(Bn, E, M, 3)
        ensemble = physics.rk4_step(ensemble, beta_h, d0, cfg.dt)
        ensemble = (ensemble + torch.randn_like(ensemble) * process_scale[None, None]).clamp_min(0.0)
        I_ens = ensemble[..., 1:4]
        means.append(I_ens.mean(dim=1))
        variances.append(I_ens.var(dim=1, unbiased=False).clamp_min(1.0))
    return torch.stack(means, dim=1), torch.stack(variances, dim=1)


## 9. Joint objective

$$L=\lambda_{data}L_{data}+\lambda_{eq}L_{eq}+\lambda_{para}L_{para}+\lambda_KL_K.$$

- $L_{data}$ combines scale-normalized linear and logarithmic Huber losses for the three predicted infectious components.
- $L_{eq}$ penalizes the trapezoidal equation residual for all five state components.
- $L_{para}$ penalizes changes in $\log\beta$ between adjacent forecast steps.
- $L_K$ compares neural forecasts with the ensemble Kalman teacher using variance weighting.


In [ ]:
def physics_weight(epoch, cfg: Config):
    progress = (epoch - cfg.physics_warmup_start) / max(cfg.physics_warmup_epochs, 1)
    return cfg.lambda_eq_max * min(1.0, max(0.0, progress))


def compute_losses(model, physics, batch, epoch, batch_index=0, use_kalman=True):
    x_hist, last_cases, y_future, _ = batch
    x_hist = x_hist.to(DEVICE)
    last_cases = last_cases.to(DEVICE)
    y_future = y_future.to(DEVICE)
    out = model(x_hist, last_cases)

    pred_I = out["states"][:, 1:, :, 1:4]
    case_scale = model.state_head.case_scale[None, None]
    linear_loss = F.smooth_l1_loss(pred_I / case_scale, y_future / case_scale, beta=1.0)
    log_scale = torch.as_tensor(SCALER["case_std"], dtype=torch.float32, device=pred_I.device)
    log_loss = F.smooth_l1_loss(
        torch.log1p(pred_I) / log_scale,
        torch.log1p(y_future) / log_scale,
        beta=1.0,
    )
    L_data = ((1.0 - CFG.log_data_weight) * linear_loss
              + CFG.log_data_weight * log_loss)
    L_eq, residual = physics.equation_loss(out["states"], out["beta"], out["d0"], CFG.dt)
    log_beta = torch.log(out["beta"].clamp_min(1e-8))
    L_para = (log_beta[:, 1:] - log_beta[:, :-1]).square().mean()

    L_k = torch.zeros((), device=DEVICE)
    kalman_used = use_kalman and CFG.lambda_kalman > 0 and batch_index % CFG.kalman_every_n_batches == 0
    if kalman_used:
        teacher_mean, teacher_var = ensemble_kalman_teacher(
            physics, out["states"][:, 0].detach(), last_cases,
            out["beta"].detach(), out["d0"].detach(), CFG,
        )
        # Use case_scale^2 as a variance floor to avoid excessively large teacher gradients.
        kalman_variance = teacher_var + case_scale.square()
        L_k = ((pred_I - teacher_mean).square() / kalman_variance).mean()

    lam_eq = physics_weight(epoch, CFG)
    total = (CFG.lambda_data * L_data + lam_eq * L_eq
             + CFG.lambda_para * L_para + CFG.lambda_kalman * L_k)
    losses = {
        "total": total, "data": L_data, "eq": L_eq,
        "para": L_para, "kalman": L_k,
        "lambda_eq": torch.tensor(lam_eq, device=DEVICE),
    }
    return out, losses


## 10. Model construction and structural checks

The optional smoke test checks output shapes, nonnegative states, positive $\beta$ and $d_0$, symmetric adjacency, finite losses, and finite backpropagation gradients. Set smoke_test=True to enable it. This check does not perform full training.


In [ ]:
def make_model():
    return PhysicsGNN(
        CFG, input_dim=FEATURES.shape[-1], base_W=W_PHYS,
        case_scale=torch.as_tensor(SCALER["case_scale"], dtype=torch.float32),
    ).to(DEVICE)


# Seeds are generated and recorded separately in the training loop.
MODEL = make_model()

def trainable_parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameter count:", f"{trainable_parameter_count(MODEL):,}")
print("Fixed physical parameters included in optimizer:", any(name in dict(MODEL.named_parameters()) for name in ["B", "mu", "omega", "alpha", "gamma", "q"]))

if CFG.smoke_test:
    smoke_batch = next(iter(train_loader))
    smoke_batch = tuple(x[:2] if torch.is_tensor(x) else x for x in smoke_batch)
    MODEL.train()
    out, losses = compute_losses(MODEL, PHYSICS, smoke_batch, epoch=CFG.physics_warmup_start + 1, batch_index=0, use_kalman=True)
    print("states:", tuple(out["states"].shape), "expected [B,P+1,7,5]")
    print("beta:", tuple(out["beta"].shape), "expected [B,P+1,7,3]")
    print("adjacency:", tuple(out["adjacency"].shape), "history-dependent graph")
    print("d0:", float(out["d0"].detach()), "| graph mix:", float(out["graph_mix"].detach()))
    print("Losses:", {k: float(v.detach()) for k, v in losses.items()})
    assert out["states"].shape[1:] == (CFG.horizon + 1, len(STATES), 5)
    assert out["beta"].shape[1:] == (CFG.horizon + 1, len(STATES), len(STRAINS))
    assert torch.isfinite(losses["total"])
    assert (out["states"] >= 0).all() and (out["beta"] > 0).all() and out["d0"] > 0
    assert torch.allclose(out["adjacency"], out["adjacency"].transpose(-1, -2), atol=1e-5)
    MODEL.zero_grad(set_to_none=True)
    losses["total"].backward()
    finite_grads = all(p.grad is None or torch.isfinite(p.grad).all() for p in MODEL.parameters())
    assert finite_grads, "Non-finite gradients during backpropagation"
    MODEL.zero_grad(set_to_none=True)
    print("Backpropagation: passed")


## 11. Optimization and early stopping

The transmission-rate head and $d_0$ use the smaller physics learning rate; the remaining neural parameters use the standard neural learning rate. Validation does not assimilate future observations. Early stopping and checkpoint selection use validation data loss.

The retained defaults are 90 maximum epochs, patience 18, and 10 independently seeded runs.


In [ ]:
def build_optimizer(model, cfg: Config):
    slow_ids = set()
    slow_ids.update(id(p) for p in model.beta_head.parameters())
    slow_ids.add(id(model.raw_d0))
    slow = [p for p in model.parameters() if id(p) in slow_ids]
    normal = [p for p in model.parameters() if id(p) not in slow_ids]
    return torch.optim.AdamW([
        {"params": normal, "lr": cfg.lr_neural},
        {"params": slow, "lr": cfg.lr_physics},
    ], weight_decay=cfg.weight_decay)


def run_epoch(model, physics, loader, optimizer, epoch, training=True):
    model.train(training)
    totals = {k: 0.0 for k in ["total", "data", "eq", "para", "kalman"]}
    n = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch_index, batch in enumerate(loader):
            if training:
                optimizer.zero_grad(set_to_none=True)
            _, losses = compute_losses(
                model, physics, batch, epoch, batch_index,
                use_kalman=training,
            )
            if training:
                losses["total"].backward()
                nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                optimizer.step()
            bsize = batch[0].shape[0]
            n += bsize
            for key in totals:
                totals[key] += float(losses[key].detach()) * bsize
    return {k: v / max(n, 1) for k, v in totals.items()}


def fit(model, physics, train_loader, val_loader, cfg: Config):
    optimizer = build_optimizer(model, cfg)
    best_state, best_val, stale = None, float("inf"), 0
    history = []
    train_started = perf_counter()
    for epoch in range(cfg.epochs):
        train_log = run_epoch(model, physics, train_loader, optimizer, epoch, training=True)
        val_log = run_epoch(model, physics, val_loader, optimizer, epoch, training=False)
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_log.items()},
               **{f"val_{k}": v for k, v in val_log.items()},
               "lambda_eq": physics_weight(epoch, cfg)}
        history.append(row)
        if val_log["data"] < best_val:
            best_val = val_log["data"]
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch % 10 == 0:
            print(f"epoch={epoch:03d} train={train_log['total']:.5f} val_data={val_log['data']:.5f} lambda_eq={row['lambda_eq']:.3f}")
        if stale >= cfg.patience:
            print("Early stopping at epoch", epoch)
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed_seconds = perf_counter() - train_started
    return pd.DataFrame(history), elapsed_seconds, float(best_val)


TRAIN_HISTORY = None
TRAIN_SECONDS = None
RUNTIME = {}
TRAINED_RUNS = []
OUTPUT_DIR = PROJECT_DIR / "forecast_sikr_outputs"

if CFG.run_training:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    training_rows = []
    for run_id in range(1, CFG.random_seed_runs + 1):
        # Generate each run seed from system entropy and record it in the results.
        run_seed = secrets.randbits(32)
        seed_everything(run_seed)
        model = make_model()
        print(f"\n===== random run {run_id}/{CFG.random_seed_runs}, seed={run_seed} =====")
        history, elapsed_seconds, best_val_data = fit(model, PHYSICS, train_loader, val_loader, CFG)
        artifact = {
            "run": run_id, "seed": run_seed, "model": model,
            "history": history, "training_seconds": elapsed_seconds,
            "best_val_data": best_val_data,
        }
        TRAINED_RUNS.append(artifact)
        training_rows.append({
            "run": run_id, "seed": run_seed, "best_val_data": best_val_data,
            "epochs_completed": int(len(history)), "training_seconds": elapsed_seconds,
        })
        print(f"run={run_id} best_val_data={best_val_data:.6f}, training_seconds={elapsed_seconds:.2f}")

    # Select the exported run using validation loss, not test metrics.
    BEST_RUN = min(TRAINED_RUNS, key=lambda item: item["best_val_data"])
    MODEL = BEST_RUN["model"]
    TRAIN_HISTORY = BEST_RUN["history"]
    TRAIN_SECONDS = BEST_RUN["training_seconds"]
    TRAIN_RUN_TABLE = pd.DataFrame(training_rows)
    TRAIN_RUN_TABLE.to_csv(OUTPUT_DIR / "random_seed_training_runs.csv", index=False, encoding="utf-8-sig")
    TRAIN_HISTORY.to_csv(OUTPUT_DIR / "training_history.csv", index=False, encoding="utf-8-sig")
    torch.save(MODEL.state_dict(), OUTPUT_DIR / "best_model.pt")
    (OUTPUT_DIR / "config.json").write_text(
        json.dumps(asdict(CFG), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    RUNTIME = {
        "random_seed_runs": int(CFG.random_seed_runs),
        "selected_run_by_validation": int(BEST_RUN["run"]),
        "selected_seed": int(BEST_RUN["seed"]),
        "training_seconds_mean": float(TRAIN_RUN_TABLE["training_seconds"].mean()),
        "training_seconds_std": float(TRAIN_RUN_TABLE["training_seconds"].std(ddof=1)),
        "training_seconds_total": float(TRAIN_RUN_TABLE["training_seconds"].sum()),
    }
    (OUTPUT_DIR / "runtime.json").write_text(
        json.dumps(RUNTIME, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    display(TRAIN_RUN_TABLE)
    print("Validation-selected export run:", BEST_RUN["run"], "| seed:", BEST_RUN["seed"])
    print("Training time over 10 runs (mean +/- standard deviation):",
          f"{RUNTIME['training_seconds_mean']:.2f} ± {RUNTIME['training_seconds_std']:.2f} seconds")
else:
    print("Full training is disabled; set CFG.run_training=True and rerun all cells.")


## 12. Test predictions, metrics, and runtime

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    predictions, targets, betas, origins = [], [], [], []
    for x_hist, last_cases, y_future, origin in loader:
        out = model(x_hist.to(DEVICE), last_cases.to(DEVICE))
        predictions.append(out["states"][:, 1:, :, 1:4].cpu().numpy())
        targets.append(y_future.numpy())
        betas.append(out["beta"][:, 1:].cpu().numpy())
        origins.append(origin.numpy())
    return (np.concatenate(predictions), np.concatenate(targets),
            np.concatenate(betas), np.concatenate(origins))


def metric_dict(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    error = y_pred - y_true
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))
    nrmse = rmse / (np.max(y_true) - np.min(y_true) + eps)
    smape = 100.0 * np.mean(2.0 * np.abs(error) / (np.abs(y_true) + np.abs(y_pred) + eps))
    nd = np.sum(np.abs(error)) / (np.sum(np.abs(y_true)) + eps)
    mape = 100.0 * np.mean(np.abs(error) / np.maximum(np.abs(y_true), 1.0))
    centered_ss = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = np.nan if centered_ss < eps else 1.0 - np.sum(error ** 2) / centered_ss
    if np.std(y_true) < eps or np.std(y_pred) < eps:
        pcc = np.nan
    else:
        pcc = np.corrcoef(y_true, y_pred)[0, 1]
    return {
        "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2,
        "NRMSE_range": nrmse, "SMAPE_%": smape, "ND": nd,
        "MAPE_%_eps1": mape, "PCC": pcc,
    }


def evaluate_all(y_true, y_pred):
    rows = [{"scope": "overall", **metric_dict(y_true, y_pred)}]
    for k, strain in enumerate(STRAINS):
        rows.append({"scope": strain, **metric_dict(y_true[..., k], y_pred[..., k])})
    for i, state in enumerate(STATES):
        rows.append({"scope": state, **metric_dict(y_true[..., i, :], y_pred[..., i, :])})
    return pd.DataFrame(rows)


if CFG.run_training:
    metric_columns = ["MAE", "MSE", "RMSE", "R2", "NRMSE_range", "SMAPE_%", "ND", "MAPE_%_eps1", "PCC"]
    trial_metrics = []
    test_runtime_rows = []
    for artifact in TRAINED_RUNS:
        test_started = perf_counter()
        pred, true, beta, origins = predict(artifact["model"], test_loader)
        metrics = evaluate_all(true, pred)
        test_seconds = perf_counter() - test_started
        metrics.insert(0, "seed", artifact["seed"])
        metrics.insert(0, "run", artifact["run"])
        trial_metrics.append(metrics)
        test_runtime_rows.append({
            "run": artifact["run"], "seed": artifact["seed"],
            "test_inference_and_metrics_seconds": test_seconds,
        })
        artifact.update({"pred": pred, "true": true, "beta": beta, "origins": origins,
                         "metrics": metrics.drop(columns=["run", "seed"]),
                         "test_seconds": test_seconds})

    TRIAL_METRICS = pd.concat(trial_metrics, ignore_index=True)
    TEST_RUNTIME_TABLE = pd.DataFrame(test_runtime_rows)
    metric_summary = TRIAL_METRICS.groupby("scope", sort=False)[metric_columns].agg(["mean", "std"])
    METRIC_SUMMARY = metric_summary.copy()
    METRIC_SUMMARY.columns = [f"{metric}_{stat}" for metric, stat in METRIC_SUMMARY.columns]
    METRIC_SUMMARY = METRIC_SUMMARY.reset_index()
    TRIAL_METRICS.to_csv(OUTPUT_DIR / "random_seed_trial_metrics.csv", index=False, encoding="utf-8-sig")
    METRIC_SUMMARY.to_csv(OUTPUT_DIR / "random_seed_metric_summary.csv", index=False, encoding="utf-8-sig")
    TEST_RUNTIME_TABLE.to_csv(OUTPUT_DIR / "random_seed_test_runtime.csv", index=False, encoding="utf-8-sig")

    RUNTIME.update({
        "test_inference_and_metrics_seconds_mean": float(TEST_RUNTIME_TABLE["test_inference_and_metrics_seconds"].mean()),
        "test_inference_and_metrics_seconds_std": float(TEST_RUNTIME_TABLE["test_inference_and_metrics_seconds"].std(ddof=1)),
        "test_inference_and_metrics_seconds_total": float(TEST_RUNTIME_TABLE["test_inference_and_metrics_seconds"].sum()),
        "total_recorded_seconds": float(RUNTIME["training_seconds_total"] + TEST_RUNTIME_TABLE["test_inference_and_metrics_seconds"].sum()),
    })
    (OUTPUT_DIR / "runtime.json").write_text(
        json.dumps(RUNTIME, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    # Export details for the validation-selected run.
    BEST_RUN = next(item for item in TRAINED_RUNS if item["run"] == RUNTIME["selected_run_by_validation"])
    TEST_PRED, TEST_TRUE = BEST_RUN["pred"], BEST_RUN["true"]
    TEST_BETA, TEST_ORIGINS_USED = BEST_RUN["beta"], BEST_RUN["origins"]
    TEST_METRICS = BEST_RUN["metrics"]
    TEST_METRICS.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False, encoding="utf-8-sig")

    rows = []
    for n, origin in enumerate(TEST_ORIGINS_USED):
        for h in range(CFG.horizon):
            target_date = DATES[int(origin) + h + 1]
            for i, state in enumerate(STATES):
                for k, strain in enumerate(STRAINS):
                    rows.append({
                        "origin_date": DATES[int(origin)], "target_date": target_date,
                        "horizon": h + 1, "state": state, "strain": strain,
                        "y_true": float(TEST_TRUE[n, h, i, k]),
                        "y_pred": float(TEST_PRED[n, h, i, k]),
                        "beta": float(TEST_BETA[n, h, i, k]),
                    })
    TEST_DETAIL = pd.DataFrame(rows)
    TEST_DETAIL.to_csv(OUTPUT_DIR / "test_predictions_and_beta.csv", index=False, encoding="utf-8-sig")
    np.savez_compressed(
        OUTPUT_DIR / "test_arrays.npz", pred=TEST_PRED, true=TEST_TRUE,
        beta=TEST_BETA, origins=TEST_ORIGINS_USED,
    )
    display(METRIC_SUMMARY)
    print("Metric means and standard deviations saved to:", OUTPUT_DIR / "random_seed_metric_summary.csv")
    print("Test inference time over 10 runs (mean +/- standard deviation):",
          f"{RUNTIME['test_inference_and_metrics_seconds_mean']:.3f} ± "
          f"{RUNTIME['test_inference_and_metrics_seconds_std']:.3f} seconds")
else:
    print("After training, this cell computes metric means and standard deviations over all 10 runs.")
